### Perf (Curves and Grid): 8020-intact animals

In [ ]:
import numpy as np
import seaborn as sns
from matplotlib.colors import Normalize
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    TIER_TITLES,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)

# Rebuild if stale:
# from mab_builders.build_all import build
# build("perf_tier_expert", "abstract_perf_probability_matrix", n_jobs=8)
gd = mab_subjects.GroupData()

# learning curves: intact sessions trimmed from the expertise day
curve_df = select(gd.perf_tier_expert.latest, paradigm="8020", lesion="intact")
curve_df = curve_df[curve_df["n_sessions"] >= 10]  # min sessions per tier per animal

# prob grids: mean P(High) over the last 90 trials of each combination (Cosyne product)
grid_df = select(
    gd.abstract_perf_probability_matrix.latest, paradigm="8020", lesion="intact"
)
probs, means, counts = prob_grid_group_means(grid_df, "perf_mat")
means["diff"] = means["struc"] - means["unstruc"]  # NaN unless both groups saw the pair

# keep only probability levels either group saw
seen = np.zeros(len(probs), dtype=bool)
for g in ["unstruc", "struc"]:
    seen |= np.isfinite(means[g]).any(axis=0) | np.isfinite(means[g]).any(axis=1)
probs = probs[seen]
means = {k: m[np.ix_(seen, seen)] for k, m in means.items()}

palette = Palette2Arm().as_dict()
hue_order = ["unstruc", "struc"]
bottom_tiers = ["low-low", "high-low", "high-high", "dependent", "independent"]

fig = plotting.Fig(2, 5, size=(18, 8), fontsize=14, axis_lw=1.5, tick_size=5)


def plot_curve(ax, tier, ylabel):
    sns.lineplot(
        data=curve_df[curve_df["tier_label"] == tier],
        x="trial_id",
        y="tier_perf",
        hue="group",
        hue_order=hue_order,
        palette=palette,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.set_ylim(0.4, 1)
    ax.set_yticks([0.4, 0.6, 0.8, 1.0])  # first and last ticks sit on the axis limits
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_ylabel("P(High)" if ylabel else "")
    ax.set_xlabel("Trials")
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_title(TIER_TITLES[tier])


# ---- top row: overall curve + prob grids (unstruc, struc, S - U) ----
plot_curve(fig.subplot(fig.gs[0, 0]), "all", ylabel=True)

grid_gs = fig.gs[0, 1:].subgridspec(1, 3)
group_norm = Normalize(  # shared scale so unstruc vs struc is comparable
    vmin=np.floor(np.nanmin([np.nanmin(means[g]) for g in hue_order]) * 20) / 20,
    vmax=np.ceil(np.nanmax([np.nanmax(means[g]) for g in hue_order]) * 20) / 20,
)
grid_axes = []
for i, g in enumerate(hue_order):
    ax = fig.subplot(grid_gs[i])
    im_group = draw_prob_grid(
        ax, means[g], probs, has_diag=False, cmap="mako", norm=group_norm
    )
    ax.set_title(f"{g} (n={counts[g]})", color=palette[g], fontweight="bold")
    if i > 0:
        ax.set_ylabel("")
    grid_axes.append(ax)
fig.fig.colorbar(im_group, ax=grid_axes, label="P(High)", shrink=0.8)

ax = fig.subplot(grid_gs[2])
im_diff = draw_prob_grid(
    ax,
    means["diff"],
    probs,
    has_diag=False,  # P(High) is undefined for equal probabilities
    **diverging_bins(np.nanmax(np.abs(means["diff"])), step=BIN_STEPS["perf_mat"]),
)
ax.set_title("struc − unstruc")
ax.set_ylabel("")
fig.fig.colorbar(im_diff, ax=ax, label="Diff. P(High)", shrink=0.8)

# ---- bottom row: learning curves per probability tier ----
for i, tier in enumerate(bottom_tiers):
    plot_curve(fig.subplot(fig.gs[1, i]), tier, ylabel=i == 0)

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "perf_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "perf_8020_intact", format="svg")

### SWP by block

In [ ]:
import mab_subjects
from mab_builders.build_all import build

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)
# abstract_swp_around_flip already exists from the Cosyne notebook; add it here if stale
build("swp_by_block_expert", exps=exps, n_jobs=6)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]

# P(Switch) per trial: all blocks, first and second block of each window
block_df = select(gd.swp_by_block_expert.latest, paradigm="8020", lesion="intact")

# P(Switch) from the previous block's last 10 trials into the new block's first 50,
# split by whether the better port flipped or stayed on the same port
flip_df = select(gd.abstract_swp_around_flip.latest, paradigm="8020", lesion="intact")

# prob grid: P(Switch) averaged over trials 2-100 of each combination (Cosyne product)
grid_df = select(
    gd.abstract_swp_probability_matrix.latest, paradigm="8020", lesion="intact"
)
probs, grid_means, grid_counts = prob_grid_group_means(grid_df, "swp_mat")
swp_diff = grid_means["struc"] - grid_means["unstruc"]  # NaN unless both saw the pair
seen = np.isfinite(swp_diff).any(axis=0) | np.isfinite(swp_diff).any(axis=1)
probs, swp_diff = probs[seen], swp_diff[np.ix_(seen, seen)]

# each animal's peak flip response after the transition, as in the Cosyne abstract:
# P(Switch | flipped) - P(Switch | same), gaussian-smoothed; its trial and height
sigma = 2  # smoothing (trials)
rows = []
for (group, name), d in flip_df[flip_df["trial_id"] > 0].groupby(["group", "name"]):
    wide = d.pivot_table(
        index="trial_id", columns="prev_best_arm", values="switch_prob"
    )
    if not {"flipped", "same"} <= set(wide.columns):
        continue
    smooth = gaussian_filter1d(
        np.nan_to_num((wide["flipped"] - wide["same"]).to_numpy()), sigma
    )
    rows.append(
        dict(
            kind="Animals",
            group=group,
            name=name,
            peak_trial=wide.index.to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 4, size=(17, 8), fontsize=14, axis_lw=1.5, tick_size=5)


def plot_swp(ax, data, title):
    sns.lineplot(
        data=data,
        x="trial_id",
        y="switch_prob",
        hue="group",
        hue_order=hue_order,
        palette=palette,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel("")


# ---- top row: P(Switch) over all blocks, block 1, block 2 (same width as bottom panels) ----
top_axes = []
for i, (label, title) in enumerate(
    [("all", "All blocks"), ("block1", "Block 1"), ("block2", "Block 2")]
):
    ax = fig.subplot(fig.gs[0, i])
    plot_swp(ax, block_df[block_df["block_label"] == label], title)
    ax.set_xlim(1, 100)
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_xlabel("Trials")
    top_axes.append(ax)

ax = fig.subplot(fig.gs[0, 3])
im = draw_prob_grid(
    ax,
    swp_diff,
    probs,
    has_diag=np.isfinite(
        np.diag(swp_diff)
    ).any(),  # switching is defined on the diagonal
    **diverging_bins(np.nanmax(np.abs(swp_diff)), step=BIN_STEPS["swp_mat"]),
)
ax.set_title(f"struc − unstruc (n={grid_counts['struc']} vs {grid_counts['unstruc']})")
fig.fig.colorbar(im, ax=ax, label="Diff. P(Switch)", shrink=0.8)

# ---- bottom row: around the block transition, flipped / same ----
bottom_axes = []
for i, (data, title) in enumerate(
    [
        (flip_df[flip_df["prev_best_arm"] == "flipped"], "Better port flipped"),
        (flip_df[flip_df["prev_best_arm"] == "same"], "Better port same"),
    ]
):
    ax = fig.subplot(fig.gs[1, i])
    plot_swp(ax, data, title)
    ax.axvline(0, color="gray", ls=":", lw=1.5)  # block transition
    ax.set_xlim(-10, 50)
    ax.set_xticks([-10, 0, 10, 20, 30, 40, 50])
    ax.set_xlabel("Trials from transition")
    bottom_axes.append(ax)

# shared y-scale within each row; bottom and top ticks sit on the axis limits
for axes in [top_axes, bottom_axes]:
    hi = max(ax.dataLim.y1 for ax in axes)
    for ax in axes:
        three_yticks(ax, lo=0.0, hi=hi)
top_axes[0].set_ylabel("P(Switch)")
bottom_axes[0].set_ylabel("P(Switch)")


# ---- flip response peak per animal, trial and height: boxplot + Mann-Whitney ----
def plot_peak_box(ax, y, ylabel, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=peaks,
            x="kind",
            y=y,
            hue="group",
            hue_order=hue_order,
            ax=ax,
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=7,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_xticks([])
    ax.set_ylabel(ylabel)


plot_peak_box(
    fig.subplot(fig.gs[1, 2]),
    "peak_swp",
    "Flip response\npeak Diff. P(Switch)",
    lambda ax: three_yticks(ax, lo=0.0),
)

plot_peak_box(
    fig.subplot(fig.gs[1, 3]),
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)
n = peaks.groupby("group")["name"].nunique()
print(n.to_dict())
print(peaks.groupby("group")[["peak_trial", "peak_swp"]].median().to_dict())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "swp_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "swp_8020_intact", format="svg")

### Early-block P(High) vs chance: high-high blocks, 8020-intact animals

In [ ]:
import seaborn as sns
from scipy.stats import ttest_1samp, wilcoxon
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm

palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
chance = 0.5
first_n = 4

# P(High) per trial in high-high blocks (both arms >= 0.5); intact sessions from the
# expertise day
df = select(
    mab_subjects.GroupData().perf_tier_expert.latest, paradigm="8020", lesion="intact"
)
df = df[
    (df["tier_label"] == "high-high")
    & (df["n_sessions"] >= 10)  # min high-high sessions per animal
    & df["trial_id"].between(1, first_n)
]
early = df.groupby(["group", "name"], as_index=False)["tier_perf"].mean()

fig = plotting.Fig(1, 3, size=(10, 4), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- per trial ----
ax = fig.subplot(fig.gs[0, 0:2])
sns.lineplot(
    data=df,
    x="trial_id",
    y="tier_perf",
    hue="group",
    hue_order=hue_order,
    palette=palette,
    lw=2.5,
    marker="o",
    markersize=8,
    err_kws=dict(edgecolor="none", alpha=0.3),
    errorbar="se",
    legend=False,
    ax=ax,
)
ax.axhline(chance, color="gray", ls="--", lw=1.5)  # chance
ax.set_xticks(range(1, first_n + 1))
ax.set_xlim(0.8, first_n + 0.2)
ax.set_ylim(0.3, 0.7)
ax.set_yticks([0.3, 0.5, 0.7])
ax.grid(axis="y", zorder=-1, alpha=0.5)
ax.set_xlabel("Trial in block")
ax.set_ylabel("P(High)")
ax.set_title(f"First {first_n} trials of high-high blocks")

# ---- per animal mean over the first trials, one-sample test against chance ----
ax = fig.subplot(fig.gs[0, 2])
sns.boxplot(
    data=early,
    x="group",
    y="tier_perf",
    hue="group",
    order=hue_order,
    hue_order=hue_order,
    palette=palette,
    width=0.5,
    showfliers=False,
    linecolor="k",
    linewidth=1,
    legend=False,
    ax=ax,
)
sns.stripplot(
    data=early,
    x="group",
    y="tier_perf",
    hue="group",
    order=hue_order,
    hue_order=hue_order,
    palette=palette_dots,
    size=7,
    jitter=0.08,
    edgecolor="w",
    linewidth=0.8,
    legend=False,
    rasterized=True,  # vector scatter markers crash Affinity
    ax=ax,
)
ax.axhline(chance, color="gray", ls="--", lw=1.5)  # chance
ax.set_ylim(0.3, 0.7)
ax.set_yticks([0.3, 0.5, 0.7])
ax.set_xlabel("")
ax.set_ylabel(f"P(High), trials 1-{first_n}")

for x, g in enumerate(hue_order):
    vals = early.loc[early["group"] == g, "tier_perf"]
    t = ttest_1samp(vals, chance)
    w = wilcoxon(vals - chance)
    p = t.pvalue
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.annotate(
        stars,
        xy=(x, 1.0),
        xycoords=("data", "axes fraction"),
        ha="center",
        va="bottom",
        fontsize=14,
    )
    print(
        f"{g} (n={len(vals)}): mean {vals.mean():.3f}; vs {chance}: "
        f"t({len(vals) - 1}) = {t.statistic:.2f}, p = {p:.2g}; Wilcoxon p = {w.pvalue:.2g}"
    )

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "perf_first5_highhigh_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "perf_first5_highhigh_8020_intact", format="svg")

### P(High) in low-low blocks, trials 10-25: struc vs unstruc, 8020-intact animals

In [ ]:
import seaborn as sns
from scipy.stats import mannwhitneyu
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from statplotannot.plots import SeabornPlotter

palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
window = (12, 19)  # trials compared (inclusive)

# P(High) per trial in low-low blocks (both arms < 0.5); intact sessions from the
# expertise day
df = select(
    mab_subjects.GroupData().perf_tier_expert.latest, paradigm="8020", lesion="intact"
)
df = df[(df["tier_label"] == "low-low") & (df["n_sessions"] >= 10)]  # min sessions
in_window = df[df["trial_id"].between(*window)]
means = in_window.groupby(["group", "name"], as_index=False)["tier_perf"].mean()
means["kind"] = "Animals"

fig = plotting.Fig(1, 3, size=(10, 4), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- per trial, compared window shaded ----
ax = fig.subplot(fig.gs[0, 0:2])
ax.axvspan(*window, color="0.9", zorder=0)
sns.lineplot(
    data=df,
    x="trial_id",
    y="tier_perf",
    hue="group",
    hue_order=hue_order,
    palette=palette,
    lw=2.5,
    err_kws=dict(edgecolor="none", alpha=0.3),
    errorbar="se",
    legend=False,
    ax=ax,
)
ax.axhline(0.5, color="gray", ls="--", lw=1.5)  # chance
ax.set_xlim(1, 100)
ax.set_xticks([1, window[0], window[1], 50, 75, 100])
ax.set_ylim(0.4, 1)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
ax.grid(axis="y", zorder=-1, alpha=0.5)
ax.set_xlabel("Trials")
ax.set_ylabel("P(High)")
ax.set_title("Low-low blocks")

# ---- per animal mean over the window: boxplot + Mann-Whitney ----
ax = fig.subplot(fig.gs[0, 2])
plotter = (
    SeabornPlotter(
        data=means,
        x="kind",
        y="tier_perf",
        hue="group",
        hue_order=hue_order,
        ax=ax,
    )
    .boxplot_filled(palette=palette, sep=0.4)
    .stripplot(
        palette=palette_dots,
        alpha=0.8,
        size=7,
        jitter=0.03,
        offset=0.1,  # dots beside, not on, their box
        edgecolor="w",
        rasterized=True,  # vector scatter markers crash Affinity
    )
)
ax.set_ylim(0.4, 1)  # set before the brackets (data units)
ax.set_yticks([0.4, 0.6, 0.8, 1.0])
plotter.stat_classical(
    "Mann-Whitney",
    pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
    loc="outside",
)
if ax.get_legend() is not None:
    ax.get_legend().remove()
ax.set_xlabel("")
ax.set_xticks([])
ax.set_ylabel(f"P(High), trials {window[0]}-{window[1]}")

u, s = (means.loc[means["group"] == g, "tier_perf"] for g in hue_order)
mw = mannwhitneyu(s, u)
print(
    f"unstruc (n={len(u)}): mean {u.mean():.3f}; struc (n={len(s)}): mean {s.mean():.3f}; "
    f"Mann-Whitney U = {mw.statistic:.0f}, p = {mw.pvalue:.2g}"
)

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "perf_lowlow_trials10-25_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "perf_lowlow_trials10-25_8020_intact", format="svg")

### P(Switch) by block type: struc vs unstruc, 8020-intact animals

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import mannwhitneyu, false_discovery_control
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import TIER_TITLES, three_yticks
from statplotannot.plots import SeabornPlotter

palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
tiers = ["low-low", "high-low", "high-high", "dependent", "independent"]
trials = (2, 20)  # trial 1 has no previous choice

# P(Switch) per trial for each block type; intact sessions from the expertise day
# (Cosyne product, tiers with < 3 sessions per animal left out)
df = select(
    mab_subjects.GroupData().abstract_swp_by_trial.latest,
    paradigm="8020",
    lesion="intact",
)
df = df[(df["outcome"] == "all") & df["tier_label"].isin(tiers)]
means = (
    df[df["trial_id"].between(*trials)]
    .groupby(["tier_label", "group", "name"], as_index=False)["switch_prob"]
    .mean()
)
# fixed box order (SeabornPlotter ignores an order= kwarg)
means["tier_label"] = pd.Categorical(means["tier_label"], categories=tiers)

# struc vs unstruc per block type, Mann-Whitney, BH-corrected across block types
pvalues = []
for tier in tiers:
    u, s = (
        means.loc[(means["tier_label"] == tier) & (means["group"] == g), "switch_prob"]
        for g in hue_order
    )
    p = mannwhitneyu(s, u).pvalue
    pvalues.append(p)
    print(
        f"{tier:12s} unstruc {u.mean():.3f} (n={len(u)}), struc {s.mean():.3f} "
        f"(n={len(s)}): Mann-Whitney p = {p:.3g}"
    )
p_bh = false_discovery_control(pvalues, method="bh")
print("BH-corrected:", dict(zip(tiers, np.round(p_bh, 4))))

fig = plotting.Fig(2, 5, size=(18, 8), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- top row: P(Switch) per trial, one panel per block type ----
curve_axes = []
for i, tier in enumerate(tiers):
    ax = fig.subplot(fig.gs[0, i])
    sns.lineplot(
        data=df[df["tier_label"] == tier],
        x="trial_id",
        y="switch_prob",
        hue="group",
        hue_order=hue_order,
        palette=palette,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_xlim(1, 100)
    ax.set_xticks([1, 50, 100])
    ax.set_xlabel("Trials")
    ax.set_ylabel("P(Switch)" if i == 0 else "")
    ax.set_title(TIER_TITLES[tier])
    curve_axes.append(ax)
hi = max(ax.dataLim.y1 for ax in curve_axes)
for ax in curve_axes:  # shared y-scale; bottom and top ticks on the axis limits
    three_yticks(ax, lo=0.0, hi=hi)

# ---- bottom row: per-animal mean P(Switch), struc vs unstruc per block type ----
ax = fig.subplot(fig.gs[1, :])
plotter = (
    SeabornPlotter(
        data=means,
        x="tier_label",
        y="switch_prob",
        hue="group",
        hue_order=hue_order,
        ax=ax,
    )
    .boxplot_filled(palette=palette, sep=0.4)
    .stripplot(
        palette=palette_dots,
        alpha=0.8,
        size=7,
        jitter=0.03,
        offset=0.1,  # dots beside, not on, their box
        edgecolor="w",
        rasterized=True,  # vector scatter markers crash Affinity
    )
)
three_yticks(ax, lo=0.0, hi=means["switch_prob"].max())  # before the brackets
plotter.stat_classical(
    pairs=[((tier, "unstruc"), (tier, "struc")) for tier in tiers],
    pvalues=list(p_bh),
    pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
    loc="outside",
)
if ax.get_legend() is not None:
    ax.get_legend().remove()
ax.set_xticks(range(len(tiers)), [TIER_TITLES[t] for t in tiers])
ax.set_xlabel("")
ax.set_ylabel(f"P(Switch)\ntrials {trials[0]}-{trials[1]}")

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "swp_by_tier_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "swp_by_tier_8020_intact", format="svg")

### Q-learning (sticky) parameters: struc vs unstruc, 8020-intact animals (all trials, high-low trials, dependent blocks)

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from scipy.stats import mannwhitneyu
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from statplotannot.plots import SeabornPlotter
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import draw_prob_grid

# 6-param sticky Q-learning (server/fit_policy_wrapper.py presets 'sticky_p8020_intact'
# and its dependent-blocks variant), 8020 intact sessions from each animal's expertise
# day; as in the Cosyne abstract. Rows: fit on all trials, on high-low trials, and on
# dependent blocks only (arm probabilities sum to 1, pairs both cohorts experience)
gd = mab_subjects.GroupData()
df = pd.concat(
    [
        gd.fit_qlearn_sticky_p8020_intact.latest,
        gd.fit_qlearn_sticky_dependent_p8020_intact.latest,
    ]
)
df = select(df, paradigm="8020", lesion="intact")
# rates and bias share the left y-axis; perseverance and beta (larger scale) the right
left_params = {
    "alpha_c": "LR\nchosen",
    "alpha_u": "LR\nunchosen",
    "alpha_h": "LR\nhabit",
    "bias": "Bias",
}
right_params = {"sticky": "Persev.", "beta": "Beta"}
param_labels = {**left_params, **right_params}
order = list(param_labels)
scopes = {
    "all": "All trials",
    "high_low": "High-low trials",
    "dependent": "Dependent blocks",
}
hue_order = ["unstruc", "struc"]
df = df[df["param_names"].isin(order)].copy()
df["param_values"] = df["param_values"].astype(float)
# fixed x positions on both y-axes (SeabornPlotter ignores an order= kwarg)
df["param_names"] = pd.Categorical(df["param_names"], categories=order)
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box


def p_to_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


# probability pairs the 8020 intact animals saw (every pair is seen by both groups)
combos = select(gd.prob_combinations.latest, paradigm="8020", lesion="intact")
combos = combos[combos["prop"] > 0][["p_high", "p_low"]].drop_duplicates()
probs = np.unique(np.r_[combos["p_high"], combos["p_low"]])
# pairs each fit used, as in mab_builders.simulate.fit_task
scope_rules = {
    "all": lambda hi, lo: True,
    "high_low": lambda hi, lo: (hi >= 0.5) + (lo >= 0.5) == 1,  # exactly one arm >= 0.5
    "dependent": lambda hi, lo: round(hi + lo, 2) == 1.0,  # probabilities sum to 1
}
used_cmap = ListedColormap(["0.88", "0.3"])  # pair seen but not used / used in the fit

# narrow left column for the pair grids, wide right column for the parameters
fig = plotting.Fig(
    3, 2, size=(16, 13), fontsize=14, axis_lw=1.5, tick_size=5, width_ratios=[1, 3]
)

for s, (scope, scope_label) in enumerate(scopes.items()):
    # ---- left: which probability pairs the fit used ----
    used = np.full((len(probs), len(probs)), np.nan)  # [higher, lower]
    for hi, lo in combos.itertuples(index=False):
        used[np.searchsorted(probs, hi), np.searchsorted(probs, lo)] = float(
            scope_rules[scope](hi, lo)
        )
    ax_grid = fig.subplot(fig.gs[s, 0])
    draw_prob_grid(ax_grid, used, probs, has_diag=False, cmap=used_cmap, vmin=0, vmax=1)
    ax_grid.set_title(f"{scope_label}\n(pairs used: dark)")

    # ---- right: fitted parameters, struc vs unstruc ----
    df_scope = df[df["fit_scope"] == scope]
    ax_left = fig.subplot(fig.gs[s, 1])
    ax_right = ax_left.twinx()

    for ax, params in ((ax_left, left_params), (ax_right, right_params)):
        SeabornPlotter(
            data=df_scope[df_scope["param_names"].isin(params)],
            x="param_names",
            y="param_values",
            hue="group",
            hue_order=hue_order,
            ax=ax,
        ).boxplot_filled(palette=palette).stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=6,
            jitter=0.04,
            offset=0.12,
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
        ax.set_xlabel("")
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        # Mann-Whitney unstruc vs struc per parameter, starred above its boxes
        ymin, ymax = ax.get_ylim()
        pad = 0.06 * (ymax - ymin)
        for param in params:
            vals = df_scope[df_scope["param_names"] == param]
            a, b = (vals[vals["group"] == g]["param_values"] for g in hue_order)
            p = mannwhitneyu(a, b).pvalue
            print(
                f"{scope:8s} {param:8s} unstruc {a.median():.3f}, struc {b.median():.3f}: p = {p:.3g}"
            )
            ax.text(
                order.index(param),
                max(a.max(), b.max()) + pad,
                p_to_stars(p),
                ha="center",
                va="bottom",
                fontsize=14,
            )
        ax.set_ylim(ymin, ymax + 3 * pad)

    ax_right.spines["right"].set_visible(True)
    ax_left.axvline(len(left_params) - 0.5, color="gray", ls=":", lw=1.5)
    ax_left.axhline(0, color="gray", lw=1, zorder=0)
    ax_left.set_xticks(range(len(order)), [param_labels[p] for p in order])
    ax_left.set_ylabel(f"{scope_label}\nLearning rate / bias")
    ax_right.set_ylabel("Persev. / beta")

print(df.groupby("group")["name"].nunique().to_dict())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "qlearn_sticky_params_8020_intact", format="png", dpi=300)
fig.savefig(figdir / "qlearn_sticky_params_8020_intact", format="svg")

### Q-learning (sticky) parameters: struc vs unstruc, 9505-intact animals (all trials, high-low trials)

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from scipy.stats import mannwhitneyu
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from statplotannot.plots import SeabornPlotter
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import draw_prob_grid

# 6-param sticky Q-learning (server/fit_policy_wrapper.py preset 'sticky_p9505'), 9505
# intact sessions kept whole (9505 data is still short, so no expertise trimming). Rows:
# fit on all trials and on high-low trials (no dependent-blocks fit for 9505)
gd = mab_subjects.GroupData()
df = select(gd.fit_qlearn_sticky_p9505.latest, paradigm="9505", lesion="intact")
# rates and bias share the left y-axis; perseverance and beta (larger scale) the right
left_params = {
    "alpha_c": "LR\nchosen",
    "alpha_u": "LR\nunchosen",
    "alpha_h": "LR\nhabit",
    "bias": "Bias",
}
right_params = {"sticky": "Persev.", "beta": "Beta"}
param_labels = {**left_params, **right_params}
order = list(param_labels)
scopes = {"all": "All trials", "high_low": "High-low trials"}
hue_order = ["unstruc", "struc"]
df = df[df["param_names"].isin(order)].copy()
df["param_values"] = df["param_values"].astype(float)
# fixed x positions on both y-axes (SeabornPlotter ignores an order= kwarg)
df["param_names"] = pd.Categorical(df["param_names"], categories=order)
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box


def p_to_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


# probability pairs the 9505 intact animals saw (any animal; not every animal saw all)
combos = select(gd.prob_combinations.latest, paradigm="9505", lesion="intact")
combos = combos[combos["prop"] > 0][["p_high", "p_low"]].drop_duplicates()
probs = np.unique(np.r_[combos["p_high"], combos["p_low"]])
# pairs each fit used, as in mab_builders.simulate.fit_task
scope_rules = {
    "all": lambda hi, lo: True,
    "high_low": lambda hi, lo: (hi >= 0.5) + (lo >= 0.5) == 1,  # exactly one arm >= 0.5
    "dependent": lambda hi, lo: round(hi + lo, 2) == 1.0,  # probabilities sum to 1
}
used_cmap = ListedColormap(["0.88", "0.3"])  # pair seen but not used / used in the fit

# narrow left column for the pair grids, wide right column for the parameters
fig = plotting.Fig(
    2, 2, size=(16, 9), fontsize=14, axis_lw=1.5, tick_size=5, width_ratios=[1, 3]
)

for s, (scope, scope_label) in enumerate(scopes.items()):
    # ---- left: which probability pairs the fit used ----
    used = np.full((len(probs), len(probs)), np.nan)  # [higher, lower]
    for hi, lo in combos.itertuples(index=False):
        used[np.searchsorted(probs, hi), np.searchsorted(probs, lo)] = float(
            scope_rules[scope](hi, lo)
        )
    ax_grid = fig.subplot(fig.gs[s, 0])
    draw_prob_grid(ax_grid, used, probs, has_diag=False, cmap=used_cmap, vmin=0, vmax=1)
    ax_grid.set_title(f"{scope_label}\n(pairs used: dark)")

    # ---- right: fitted parameters, struc vs unstruc ----
    df_scope = df[df["fit_scope"] == scope]
    ax_left = fig.subplot(fig.gs[s, 1])
    ax_right = ax_left.twinx()

    for ax, params in ((ax_left, left_params), (ax_right, right_params)):
        SeabornPlotter(
            data=df_scope[df_scope["param_names"].isin(params)],
            x="param_names",
            y="param_values",
            hue="group",
            hue_order=hue_order,
            ax=ax,
        ).boxplot_filled(palette=palette).stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=6,
            jitter=0.04,
            offset=0.12,
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
        ax.set_xlabel("")
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        # Mann-Whitney unstruc vs struc per parameter, starred above its boxes
        ymin, ymax = ax.get_ylim()
        pad = 0.06 * (ymax - ymin)
        for param in params:
            vals = df_scope[df_scope["param_names"] == param]
            a, b = (vals[vals["group"] == g]["param_values"] for g in hue_order)
            p = mannwhitneyu(a, b).pvalue
            print(
                f"{scope:8s} {param:8s} unstruc {a.median():.3f}, struc {b.median():.3f}: p = {p:.3g}"
            )
            ax.text(
                order.index(param),
                max(a.max(), b.max()) + pad,
                p_to_stars(p),
                ha="center",
                va="bottom",
                fontsize=14,
            )
        ax.set_ylim(ymin, ymax + 3 * pad)

    ax_right.spines["right"].set_visible(True)
    ax_left.axvline(len(left_params) - 0.5, color="gray", ls=":", lw=1.5)
    ax_left.axhline(0, color="gray", lw=1, zorder=0)
    ax_left.set_xticks(range(len(order)), [param_labels[p] for p in order])
    ax_left.set_ylabel(f"{scope_label}\nLearning rate / bias")
    ax_right.set_ylabel("Persev. / beta")

print(df.groupby("group")["name"].nunique().to_dict())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "qlearn_sticky_params_9505_intact", format="png", dpi=300)
fig.savefig(figdir / "qlearn_sticky_params_9505_intact", format="svg")

### P(High), P(Switch) and struc - unstruc prob grids: 8020 vs 9505, intact animals

In [ ]:
import mab_subjects
from mab_builders.build_all import build

# per-trial P(High) and P(Switch) tiers and prob grids over 8020 + 9505 good intact animals, intact
# sessions from the expertise day (separate products, so the shared ones stay whole)
exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    + mab_subjects.unstruc.p9505_good_intact_sess
    + mab_subjects.struc.p9505_good_intact_sess
)
build(
    "perf_tier_8020_9505",
    "swp_tier_8020_9505",
    "perf_probability_matrix_8020_9505",
    "swp_probability_matrix_8020_9505",
    exps=exps,
    n_jobs=6,
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from scipy.stats import mannwhitneyu
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
groups = ["unstruc", "struc"]
paradigms = ["8020", "9505"]

# cohort = paradigm + group; 8020 solid in the group colour, 9505 dashed and darker
cohort_order = [f"{p}_{g}" for g in groups for p in paradigms]
base, dark = Palette2Arm().as_dict(), Palette2Arm(lightness_scale=0.6).as_dict()
palette = {
    f"{p}_{g}": (base if p == "8020" else dark)[g] for g in groups for p in paradigms
}
dashes = {c: (4, 2) if c.startswith("9505") else "" for c in cohort_order}


def add_cohort(df):
    return df.assign(cohort=df["paradigm"] + "_" + df["group"])


# per-trial P(High) and P(Switch) over all blocks; intact sessions from the expertise day
perf = add_cohort(
    select(gd.perf_tier_8020_9505.latest, paradigm=paradigms, lesion="intact")
)
perf = perf[perf["tier_label"] == "all"]
swp = add_cohort(
    select(gd.swp_tier_8020_9505.latest, paradigm=paradigms, lesion="intact")
)
swp = swp[(swp["tier_label"] == "all") & (swp["outcome"] == "all")]

# struc - unstruc prob grids per paradigm: P(High) over the last 90 trials, P(Switch)
# over trials 2-100 of each probability pair (same expertise trimming)
grid_products = {
    "perf_mat": "perf_probability_matrix_8020_9505",
    "swp_mat": "swp_probability_matrix_8020_9505",
}
grids = {}
for mat_col, product in grid_products.items():
    df = getattr(gd, product).latest
    for par in paradigms:
        probs, means, counts = prob_grid_group_means(
            select(df, paradigm=par, lesion="intact"), mat_col
        )
        diff = means["struc"] - means["unstruc"]  # NaN unless both groups saw the pair
        seen = np.isfinite(diff).any(axis=0) | np.isfinite(diff).any(axis=1)
        grids[mat_col, par] = (probs[seen], diff[np.ix_(seen, seen)], counts)

rows = [
    # (data, value column, y label, per-animal summary window, grid column)
    (perf, "tier_perf", "P(High)", (51, 100), "perf_mat"),
    (swp, "switch_prob", "P(Switch)", (2, 100), "swp_mat"),
]

# columns: unstruc curves, struc curves, per-animal summary, S - U grid 8020, 9505
fig = plotting.Fig(2, 5, size=(25, 9), fontsize=14, axis_lw=1.5, tick_size=5)
summary_axes = []

for r, (data, y, ylabel, window, mat_col) in enumerate(rows):
    # ---- curves: 8020 vs 9505, one panel per group ----
    curve_axes = []
    for c, g in enumerate(groups):
        ax = fig.subplot(fig.gs[r, c])
        ax.axvspan(*window, color="0.93", zorder=0)  # summarised trials
        sns.lineplot(
            data=data[data["group"] == g],
            x="trial_id",
            y=y,
            hue="cohort",
            hue_order=[f"{p}_{g}" for p in paradigms],
            palette=palette,
            style="cohort",
            style_order=[f"{p}_{g}" for p in paradigms],
            dashes=dashes,
            lw=2.5,
            err_kws=dict(edgecolor="none", alpha=0.3),
            errorbar="se",
            legend=False,
            ax=ax,
        )
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        ax.set_xlim(1, 100)
        ax.set_xticks([1, 25, 50, 75, 100])
        ax.set_xlabel("Trials")
        ax.set_ylabel(ylabel if c == 0 else "")
        if r == 0:
            n = data[data["group"] == g].groupby("paradigm")["name"].nunique()
            ax.set_title(f"{g} (n = {n.get('8020', 0)} / {n.get('9505', 0)})")
        curve_axes.append(ax)
    if y == "tier_perf":
        for ax in curve_axes:
            ax.set_ylim(0.4, 1)
            ax.set_yticks([0.4, 0.6, 0.8, 1.0])  # first and last ticks on the limits
    else:
        hi = max(ax.dataLim.y1 for ax in curve_axes)
        for ax in curve_axes:
            three_yticks(ax, lo=0.0, hi=hi)

    # ---- per animal mean over the window: 8020 vs 9505 within each group ----
    means = (
        data[data["trial_id"].between(*window)]
        .groupby(["cohort", "paradigm", "group", "name"], as_index=False)[y]
        .mean()
        .assign(kind="")
    )
    pairs, pvalues = [], []
    for g in groups:
        a, b = (means.loc[means["cohort"] == f"{p}_{g}", y] for p in paradigms)
        p = mannwhitneyu(a, b).pvalue
        pairs.append((("", f"8020_{g}"), ("", f"9505_{g}")))
        pvalues.append(p)
        print(
            f"{ylabel} trials {window[0]}-{window[1]}, {g}: 8020 {a.mean():.3f} (n={len(a)}) "
            f"vs 9505 {b.mean():.3f} (n={len(b)}), Mann-Whitney p = {p:.3g}"
        )
    ax = fig.subplot(fig.gs[r, 2])
    plotter = (
        SeabornPlotter(
            data=means, x="kind", y=y, hue="cohort", hue_order=cohort_order, ax=ax
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette,
            alpha=0.9,
            size=6,
            jitter=0.03,
            offset=0.08,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    if y == "tier_perf":
        ax.set_ylim(0.4, 1)  # set before the brackets (data units)
        ax.set_yticks([0.4, 0.6, 0.8, 1.0])
    else:
        three_yticks(ax, lo=0.0, hi=means[y].max())
    plotter.stat_classical(
        pairs=pairs,
        pvalues=pvalues,
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_xticks([])
    ax.set_ylabel(f"{ylabel}\ntrials {window[0]}-{window[1]}")
    summary_axes.append(ax)

    # ---- struc - unstruc prob grids, 8020 and 9505 on one colour scale ----
    colors = diverging_bins(
        np.nanmax([np.nanmax(np.abs(grids[mat_col, par][1])) for par in paradigms]),
        step=BIN_STEPS[mat_col],
    )
    grid_axes = []
    for i, par in enumerate(paradigms):
        probs, diff, counts = grids[mat_col, par]
        ax = fig.subplot(fig.gs[r, 3 + i])
        im = draw_prob_grid(
            ax, diff, probs, has_diag=np.isfinite(np.diag(diff)).any(), **colors
        )
        ax.set_title(
            f"{ylabel}: struc − unstruc\n{par} (n={counts['struc']} vs {counts['unstruc']})"
        )
        if i > 0:
            ax.set_ylabel("")
        grid_axes.append(ax)
    # colorbar at the right end of the row, ticks at the end blocks' centres and 0
    cb = fig.fig.colorbar(im, ax=grid_axes, label=f"Diff. {ylabel}", shrink=0.8)
    edges = im.norm.boundaries
    end = (edges[-2] + edges[-1]) / 2
    cb.set_ticks([-end, 0, end])
    cb.ax.minorticks_off()

# legend under the lower summary panel
summary_axes[-1].legend(
    [
        Line2D([], [], color=palette[c], lw=3, dashes=dashes[c] or (None, None))
        for c in cohort_order
    ],
    [c.replace("_", " ") for c in cohort_order],
    ncol=2,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
)

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "perf_swp_8020_vs_9505_intact", format="png", dpi=300)
fig.savefig(figdir / "perf_swp_8020_vs_9505_intact", format="svg")

### RNN vs Intact

In [ ]:
import mab_subjects
from mab_builders.build_all import build

exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
    + mab_subjects.unstruc_rnn.p8020_good_sess
    + mab_subjects.struc_rnn.p8020_good_sess
)
# the prob-grid _rnn products already exist from the Cosyne notebook; add them if stale
build("abstract_swp_around_flip_rnn", exps=exps, n_jobs=6)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
kinds = {"animal": "Animals", "rnn": "RNNs"}


def animal_and_rnn(df):
    """8020 intact animals and 8020 RNN models, tagged by 'kind'."""
    return pd.concat(
        [
            select(df, paradigm="8020", lesion="intact").assign(kind="animal"),
            select(df, paradigm="8020", rnn=True).assign(kind="rnn"),
        ]
    )


def s_minus_u_grids(df, mat_col):
    """struc - unstruc grid per kind, on probability axes shared by both kinds."""
    df = df.assign(group=df["group"] + "_" + df["kind"])
    probs, means, counts = prob_grid_group_means(
        df, mat_col, groups=[f"{g}_{k}" for k in kinds for g in hue_order]
    )
    diffs = {k: means[f"struc_{k}"] - means[f"unstruc_{k}"] for k in kinds}
    seen = np.zeros(len(probs), dtype=bool)
    for d in diffs.values():
        seen |= np.isfinite(d).any(axis=0) | np.isfinite(d).any(axis=1)
    diffs = {k: d[np.ix_(seen, seen)] for k, d in diffs.items()}
    return probs[seen], diffs, counts


# prob grids (Cosyne products): P(High) over the last 90 trials, P(Switch) over trials 2-100
perf_grids = s_minus_u_grids(
    animal_and_rnn(gd.abstract_perf_probability_matrix_rnn.latest), "perf_mat"
)
swp_grids = s_minus_u_grids(
    animal_and_rnn(gd.abstract_swp_probability_matrix_rnn.latest), "swp_mat"
)

# flip response: P(Switch | flipped) - P(Switch | same), from the previous block's
# last 10 trials into the new block's first 50
flip_df = animal_and_rnn(gd.abstract_swp_around_flip_rnn.latest)
wide = flip_df.pivot_table(
    index=["kind", "group", "name", "trial_id"],
    columns="prev_best_arm",
    values="switch_prob",
)
resp_df = (wide["flipped"] - wide["same"]).rename("flip_resp").reset_index()

# each subject's peak flip response after the transition (gaussian-smoothed), as in
# the Cosyne abstract: its trial and height
sigma = 2  # smoothing (trials)
rows = []
for (kind, group, name), d in resp_df[resp_df["trial_id"] > 0].groupby(
    ["kind", "group", "name"]
):
    smooth = gaussian_filter1d(np.nan_to_num(d["flip_resp"].to_numpy()), sigma)
    rows.append(
        dict(
            kind=kinds[kind],
            group=group,
            name=name,
            peak_trial=d["trial_id"].to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 4, size=(17, 9), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- top row: struc - unstruc prob grids, animals vs RNNs ----
for col, (name, (probs, diffs, counts), mat_col, label) in enumerate(
    [
        ("P(High)", perf_grids, "perf_mat", "Diff. P(High)"),
        ("P(Switch)", swp_grids, "swp_mat", "Diff. P(Switch)"),
    ]
):
    # shared color scale so the animal and RNN struc advantage are comparable
    colors = diverging_bins(
        np.nanmax([np.nanmax(np.abs(d)) for d in diffs.values()]),
        step=BIN_STEPS[mat_col],
    )
    axes = []
    for i, (k, kind_label) in enumerate(kinds.items()):
        ax = fig.subplot(fig.gs[0, 2 * col + i])
        im = draw_prob_grid(
            ax,
            diffs[k],
            probs,
            has_diag=np.isfinite(np.diag(diffs[k])).any(),  # P(High) has none
            **colors,
        )
        ax.set_title(
            f"{name}: {kind_label}\nstruc − unstruc "
            f"(n={counts[f'struc_{k}']} vs {counts[f'unstruc_{k}']})"
        )
        if i > 0:
            ax.set_ylabel("")
        axes.append(ax)
    fig.fig.colorbar(im, ax=axes, label=label, shrink=0.8)

# ---- bottom row: P(Switch) around the transition (rows: animals, RNNs; columns:
# raw flipped, raw same, flipped - same), then peak height and peak trial ----
curve_gs = fig.gs[1, 0:2].subgridspec(2, 3)
curve_cols = [
    ("flipped", flip_df[flip_df["prev_best_arm"] == "flipped"], "switch_prob"),
    ("same", flip_df[flip_df["prev_best_arm"] == "same"], "switch_prob"),
    ("flipped − same", resp_df, "flip_resp"),
]
for r, (k, kind_label) in enumerate(kinds.items()):
    row_axes = []
    for j, (title, data, y) in enumerate(curve_cols):
        ax = fig.subplot(curve_gs[r, j])
        sns.lineplot(
            data=data[data["kind"] == k],
            x="trial_id",
            y=y,
            hue="group",
            hue_order=hue_order,
            palette=palette,
            lw=2,
            err_kws=dict(edgecolor="none", alpha=0.3),
            errorbar="se",
            legend=False,
            ax=ax,
        )
        ax.axvline(0, color="gray", ls=":", lw=1.2)  # block transition
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        ax.set_xlim(-10, 50)
        ax.set_xticks([-10, 20, 50])  # transition marked by the dotted line
        ax.set_xlabel("Trials from transition" if r == 1 else "")
        ax.set_ylabel(f"{kind_label}\nP(Switch)" if j == 0 else "")
        if r == 0:
            ax.set_title(title)
        row_axes.append(ax)

    # raw flipped and same share a y-scale; ticks at 0, middle, top
    hi = max(ax.dataLim.y1 for ax in row_axes[:2])
    for ax in row_axes[:2]:
        three_yticks(ax, lo=0.0, hi=hi)
    ax = row_axes[2]
    ax.axhline(0, color="gray", lw=1)
    top = three_yticks(ax, lo=0.0)[-1]
    ax.set_ylim(-0.1 * top, top)  # room for small dips below 0


def plot_peak_box(ax, data, y, ylabel, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=data,
            x="kind",
            y=y,
            hue="group",
            hue_order=hue_order,
            ax=ax,
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=5,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)


# peak height: RNNs switch ~10x more than animals, so each kind gets its own y-axis
height_gs = fig.gs[1, 2].subgridspec(1, 2)
for i, kind_label in enumerate(kinds.values()):
    plot_peak_box(
        fig.subplot(height_gs[i]),
        peaks[peaks["kind"] == kind_label],
        "peak_swp",
        "Flip response\npeak Diff. P(Switch)" if i == 0 else "",
        lambda ax: three_yticks(ax, lo=0.0),
    )

# peak trial: same trial scale for animals and RNNs
plot_peak_box(
    fig.subplot(fig.gs[1, 3]),
    peaks,
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)

print(peaks.groupby(["kind", "group"])[["peak_trial", "peak_swp"]].median())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "animal_rnn_8020", format="png", dpi=300)
fig.savefig(figdir / "animal_rnn_8020", format="svg")

### mPFC lesion: intact vs lesion (8020, same animals)

In [ ]:
import mab_subjects
from mab_builders.build_all import build

# intact (from the expertise day) + post-lesion sessions of the mPFC-lesioned animals;
# perf_tier_expert already covers them (built over all good sessions)
exps = (
    mab_subjects.unstruc.p8020_lesion_mPFC_intact_post_sess
    + mab_subjects.struc.p8020_lesion_mPFC_intact_post_sess
)
build(
    "abstract_perf_probability_matrix_lesion_mPFC",
    "abstract_swp_probability_matrix_lesion_mPFC",
    "abstract_swp_around_flip_lesion_mPFC",
    "swp_by_block_expert_lesion_mPFC",
    exps=exps,
    n_jobs=6,
)

In [ ]:
import numpy as np
import seaborn as sns
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import palette_2arm
from mab_abstract.mab_abstract_plotters import (
    TIER_TITLES,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)

gd = mab_subjects.GroupData()
lesion_tag = "lesion_mPFC_post"
groups = ["unstruc", "struc"]

# cohort = condition + group; intact solid in the group colour, lesion dashed and darker
cohorts = {
    f"{cond}_{group}": (group, tag)
    for group in groups
    for cond, tag in [("intact", "intact"), ("lesion", lesion_tag)]
}
palette = {c: palette_2arm.lesion(g)[tag] for c, (g, tag) in cohorts.items()}
dashes = {c: (4, 2) if c.startswith("lesion") else "" for c in cohorts}


def add_cohort(df):
    condition = df["lesion"].map({"intact": "intact", lesion_tag: "lesion"})
    return df.assign(cohort=condition + "_" + df["group"])


# learning curves: intact sessions from the expertise day, lesion sessions whole
curve_df = add_cohort(
    select(gd.perf_tier_expert.latest, paradigm="8020", paired_lesion=lesion_tag)
)
curve_df = curve_df[curve_df["n_sessions"] >= 10]  # min sessions per tier per animal
n = curve_df.groupby("group")["name"].nunique()

# prob grids: mean P(High) over the last 90 trials of each combination
grid_df = add_cohort(
    select(
        gd.abstract_perf_probability_matrix_lesion_mPFC.latest,
        paradigm="8020",
        paired_lesion=lesion_tag,
    )
)
probs, means, _ = prob_grid_group_means(
    grid_df.assign(group=grid_df["cohort"]), "perf_mat", groups=list(cohorts)
)
diffs = {g: means[f"lesion_{g}"] - means[f"intact_{g}"] for g in groups}
diffs["s_minus_u"] = means["lesion_struc"] - means["lesion_unstruc"]
seen = np.zeros(len(probs), dtype=bool)
for d in diffs.values():
    seen |= np.isfinite(d).any(axis=0) | np.isfinite(d).any(axis=1)
probs = probs[seen]
diffs = {k: d[np.ix_(seen, seen)] for k, d in diffs.items()}

bottom_tiers = ["low-low", "high-low", "high-high", "dependent", "independent"]

fig = plotting.Fig(2, 5, size=(18, 8), fontsize=14, axis_lw=1.5, tick_size=5)


def plot_curve(ax, tier, ylabel):
    sns.lineplot(
        data=curve_df[curve_df["tier_label"] == tier],
        x="trial_id",
        y="tier_perf",
        hue="cohort",
        hue_order=list(cohorts),
        palette=palette,
        style="cohort",
        style_order=list(cohorts),
        dashes=dashes,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.set_ylim(0.4, 1)
    ax.set_yticks([0.4, 0.6, 0.8, 1.0])  # first and last ticks sit on the axis limits
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_ylabel("P(High)" if ylabel else "")
    ax.set_xlabel("Trials")
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_title(TIER_TITLES[tier])


# ---- top row: overall curve + prob grids (lesion - intact per group, lesion S - U) ----
plot_curve(fig.subplot(fig.gs[0, 0]), "all", ylabel=True)

grid_gs = fig.gs[0, 1:].subgridspec(1, 3)
# shared scale so the unstruc and struc lesion effects are comparable
lesion_colors = diverging_bins(
    np.nanmax([np.nanmax(np.abs(diffs[g])) for g in groups]), step=BIN_STEPS["perf_mat"]
)
grid_axes = []
for i, g in enumerate(groups):
    ax = fig.subplot(grid_gs[i])
    im = draw_prob_grid(ax, diffs[g], probs, has_diag=False, **lesion_colors)
    ax.set_title(f"{g}: lesion − intact (n={n.get(g, 0)})")
    if i > 0:
        ax.set_ylabel("")
    grid_axes.append(ax)
fig.fig.colorbar(im, ax=grid_axes, label="Diff. P(High)", shrink=0.8)

ax = fig.subplot(grid_gs[2])
im = draw_prob_grid(
    ax,
    diffs["s_minus_u"],
    probs,
    has_diag=False,  # P(High) is undefined for equal probabilities
    **diverging_bins(np.nanmax(np.abs(diffs["s_minus_u"])), step=BIN_STEPS["perf_mat"]),
)
ax.set_title("Lesion: struc − unstruc")
ax.set_ylabel("")
fig.fig.colorbar(im, ax=ax, label="Diff. P(High)", shrink=0.8)

# ---- bottom row: learning curves per probability tier ----
for i, tier in enumerate(bottom_tiers):
    plot_curve(fig.subplot(fig.gs[1, i]), tier, ylabel=i == 0)

print("solid = intact, dashed = lesion;", n.to_dict())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "perf_8020_mPFC_lesion", format="png", dpi=300)
fig.savefig(figdir / "perf_8020_mPFC_lesion", format="svg")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from scipy.stats import ttest_rel
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import palette_2arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)

gd = mab_subjects.GroupData()
lesion_tag = "lesion_mPFC_post"
groups = ["unstruc", "struc"]

# cohort = condition + group; intact solid in the group colour, lesion dashed and darker
cohorts = {
    f"{cond}_{group}": (group, tag)
    for group in groups
    for cond, tag in [("intact", "intact"), ("lesion", lesion_tag)]
}
palette = {c: palette_2arm.lesion(g)[tag] for c, (g, tag) in cohorts.items()}
dashes = {c: (4, 2) if c.startswith("lesion") else "" for c in cohorts}


def add_cohort(df):
    condition = df["lesion"].map({"intact": "intact", lesion_tag: "lesion"})
    return df.assign(cohort=condition + "_" + df["group"])


def lesion_select(product):
    return add_cohort(
        select(getattr(gd, product).latest, paradigm="8020", paired_lesion=lesion_tag)
    )


# P(Switch) per trial: all blocks, first and second block of each window
block_df = lesion_select("swp_by_block_expert_lesion_mPFC")

# P(Switch) from the previous block's last 10 trials into the new block's first 50,
# split by whether the better port flipped or stayed on the same port
flip_df = lesion_select("abstract_swp_around_flip_lesion_mPFC")

# prob grid: P(Switch) averaged over trials 2-100, lesion - intact per group
grid_df = lesion_select("abstract_swp_probability_matrix_lesion_mPFC")
probs, means, _ = prob_grid_group_means(
    grid_df.assign(group=grid_df["cohort"]), "swp_mat", groups=list(cohorts)
)
diffs = {g: means[f"lesion_{g}"] - means[f"intact_{g}"] for g in groups}
seen = np.zeros(len(probs), dtype=bool)
for d in diffs.values():
    seen |= np.isfinite(d).any(axis=0) | np.isfinite(d).any(axis=1)
probs = probs[seen]
diffs = {g: d[np.ix_(seen, seen)] for g, d in diffs.items()}

# each animal's peak flip response after the transition, intact and lesion, as in the
# Cosyne abstract: P(Switch | flipped) - P(Switch | same), gaussian-smoothed
sigma = 2  # smoothing (trials)
rows = []
for (group, lesion, name), d in flip_df[flip_df["trial_id"] > 0].groupby(
    ["group", "lesion", "name"]
):
    wide = d.pivot_table(
        index="trial_id", columns="prev_best_arm", values="switch_prob"
    )
    if not {"flipped", "same"} <= set(wide.columns):
        continue
    smooth = gaussian_filter1d(
        np.nan_to_num((wide["flipped"] - wide["same"]).to_numpy()), sigma
    )
    rows.append(
        dict(
            group=group,
            lesion=lesion,
            name=name,
            peak_trial=wide.index.to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 5, size=(20, 8), fontsize=14, axis_lw=1.5, tick_size=5)


def plot_swp(ax, data, title):
    sns.lineplot(
        data=data,
        x="trial_id",
        y="switch_prob",
        hue="cohort",
        hue_order=list(cohorts),
        palette=palette,
        style="cohort",
        style_order=list(cohorts),
        dashes=dashes,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel("")


# ---- top row: P(Switch) over all blocks, block 1, block 2 ----
top_axes = []
for i, (label, title) in enumerate(
    [("all", "All blocks"), ("block1", "Block 1"), ("block2", "Block 2")]
):
    ax = fig.subplot(fig.gs[0, i])
    plot_swp(ax, block_df[block_df["block_label"] == label], title)
    ax.set_xlim(1, 100)
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_xlabel("Trials")
    top_axes.append(ax)

# lesion - intact P(Switch) grids, unstruc and struc on a shared scale
colors = diverging_bins(
    np.nanmax([np.nanmax(np.abs(d)) for d in diffs.values()]), step=BIN_STEPS["swp_mat"]
)
grid_axes = []
for i, g in enumerate(groups):
    ax = fig.subplot(fig.gs[0, 3 + i])
    im = draw_prob_grid(
        ax, diffs[g], probs, has_diag=np.isfinite(np.diag(diffs[g])).any(), **colors
    )
    ax.set_title(f"{g}: lesion − intact")
    if i > 0:
        ax.set_ylabel("")
    grid_axes.append(ax)
fig.fig.colorbar(im, ax=grid_axes, label="Diff. P(Switch)", shrink=0.8)

# ---- bottom row (4 panels across the full width): flipped / same, then peaks ----
bottom_gs = fig.gs[1, :].subgridspec(1, 4)
bottom_axes = []
for i, (arm, title) in enumerate(
    [("flipped", "Better port flipped"), ("same", "Better port same")]
):
    ax = fig.subplot(bottom_gs[i])
    plot_swp(ax, flip_df[flip_df["prev_best_arm"] == arm], title)
    ax.axvline(0, color="gray", ls=":", lw=1.5)  # block transition
    ax.set_xlim(-10, 50)
    ax.set_xticks([-10, 0, 10, 20, 30, 40, 50])
    ax.set_xlabel("Trials from transition")
    bottom_axes.append(ax)

# shared y-scale within each row; bottom and top ticks sit on the axis limits
for axes in [top_axes, bottom_axes]:
    hi = max(ax.dataLim.y1 for ax in axes)
    for ax in axes:
        three_yticks(ax, lo=0.0, hi=hi)
top_axes[0].set_ylabel("P(Switch)")
bottom_axes[0].set_ylabel("P(Switch)")

# ---- flip response peak, intact -> lesion per animal: paired t-test per group ----
# (2-3 animals per group, so one dot per animal joined by a line instead of boxes;
# filled = intact, open = lesion, bar = mean)
x_pos = {
    "intact_unstruc": 0,
    "lesion_unstruc": 1,
    "intact_struc": 2.5,
    "lesion_struc": 3.5,
}


def plot_paired(ax, y, ylabel, set_yaxis):
    set_yaxis(ax)
    for g in groups:
        paired = (
            peaks[peaks["group"] == g]
            .pivot_table(index="name", columns="lesion", values=y)[
                ["intact", lesion_tag]
            ]
            .dropna()
        )
        xs = [x_pos[f"intact_{g}"], x_pos[f"lesion_{g}"]]
        for _, row in paired.iterrows():
            ax.plot(xs, row.values, color="0.6", lw=1.2, zorder=2)
        for x, cond, tag in zip(xs, ("intact", "lesion"), ("intact", lesion_tag)):
            color = palette[f"{cond}_{g}"]
            ax.scatter(
                np.full(len(paired), x),
                paired[tag],
                s=70,
                color=color if cond == "intact" else "white",
                edgecolor=color,
                linewidth=2,
                zorder=3,
            )
            ax.hlines(
                paired[tag].mean(), x - 0.25, x + 0.25, color=color, lw=3, zorder=4
            )
        if len(paired) >= 2:
            p = ttest_rel(paired["intact"], paired[lesion_tag]).pvalue
            stars = (
                "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
            )
            print(f"{y} {g}: paired t-test p = {p:.3f} (n={len(paired)})")
            ax.annotate(
                stars,
                xy=(np.mean(xs), 1.0),
                xycoords=("data", "axes fraction"),
                ha="center",
                va="bottom",
                fontsize=14,
            )
    ax.set_xlim(-0.6, 4.1)
    ax.set_xticks([0.5, 3.0], groups)
    ax.set_ylabel(ylabel)


plot_paired(
    fig.subplot(bottom_gs[2]),
    "peak_swp",
    "Flip response\npeak Diff. P(Switch)",
    lambda ax: three_yticks(ax, lo=0.0, hi=peaks["peak_swp"].max()),
)
plot_paired(
    fig.subplot(bottom_gs[3]),
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)

print("solid / filled = intact, dashed / open = lesion")
print(peaks.groupby(["group", "lesion"])[["peak_trial", "peak_swp"]].median())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "swp_8020_mPFC_lesion", format="png", dpi=300)
fig.savefig(figdir / "swp_8020_mPFC_lesion", format="svg")

### Q-learning (sticky) parameters: intact vs mPFC lesion, 8020 (all trials, high-low trials, dependent blocks)

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from scipy.stats import ttest_rel
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import palette_2arm
from mab_abstract.mab_abstract_plotters import draw_prob_grid

# 6-param sticky Q-learning fitted before (intact, from the expertise day) and after the
# mPFC lesion (lesion sessions whole), on all trials, high-low trials and dependent
# blocks; only animals fitted both before and after the lesion (their own baseline).
# As in the Cosyne abstract's intact vs mPFC cells.
gd = mab_subjects.GroupData()
lesion_tag = "lesion_mPFC_post"
df = pd.concat(
    [
        gd.fit_qlearn_sticky_p8020_intact.latest,
        gd.fit_qlearn_sticky_p8020_lesion_mPFC.latest,
        gd.fit_qlearn_sticky_dependent_p8020_intact.latest,
        gd.fit_qlearn_sticky_dependent_p8020_lesion_mPFC.latest,
    ]
)
df = select(df, paradigm="8020", paired_lesion=lesion_tag)

# rates and bias share the left y-axis; perseverance and beta (larger scale) the right
left_params = {
    "alpha_c": "LR\nchosen",
    "alpha_u": "LR\nunchosen",
    "alpha_h": "LR\nhabit",
    "bias": "Bias",
}
right_params = {"sticky": "Persev.", "beta": "Beta"}
param_labels = {**left_params, **right_params}
order = list(param_labels)
scopes = {
    "all": "All trials",
    "high_low": "High-low trials",
    "dependent": "Dependent blocks",
}
groups = ["unstruc", "struc"]
df = df[df["param_names"].isin(order)].copy()
df["param_values"] = df["param_values"].astype(float)

# cohort = condition + group; intact filled in the group colour, lesion open and darker
cohort_order = ["intact_unstruc", "lesion_unstruc", "intact_struc", "lesion_struc"]
palette = {
    f"{cond}_{group}": palette_2arm.lesion(group)[tag]
    for group in groups
    for cond, tag in [("intact", "intact"), ("lesion", lesion_tag)]
}
# x offset of each cohort around its parameter's position
dodge = {c: -0.3 + 0.2 * k for k, c in enumerate(cohort_order)}


def p_to_stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"


# probability pairs the animals saw, and the pairs each fit used (mab_builders.simulate.fit_task)
combos = select(gd.prob_combinations.latest, paradigm="8020", paired_lesion=lesion_tag)
combos = combos[combos["prop"] > 0][["p_high", "p_low"]].drop_duplicates()
probs = np.unique(np.r_[combos["p_high"], combos["p_low"]])
scope_rules = {
    "all": lambda hi, lo: True,
    "high_low": lambda hi, lo: (hi >= 0.5) + (lo >= 0.5) == 1,  # exactly one arm >= 0.5
    "dependent": lambda hi, lo: round(hi + lo, 2) == 1.0,  # probabilities sum to 1
}
used_cmap = ListedColormap(["0.88", "0.3"])  # pair seen but not used / used in the fit

# narrow left column for the pair grids, wide right column for the parameters
fig = plotting.Fig(
    3, 2, size=(16, 13), fontsize=14, axis_lw=1.5, tick_size=5, width_ratios=[1, 3]
)

for s, (scope, scope_label) in enumerate(scopes.items()):
    # ---- left: which probability pairs the fit used ----
    used = np.full((len(probs), len(probs)), np.nan)  # [higher, lower]
    for hi, lo in combos.itertuples(index=False):
        used[np.searchsorted(probs, hi), np.searchsorted(probs, lo)] = float(
            scope_rules[scope](hi, lo)
        )
    ax_grid = fig.subplot(fig.gs[s, 0])
    draw_prob_grid(ax_grid, used, probs, has_diag=False, cmap=used_cmap, vmin=0, vmax=1)
    ax_grid.set_title(f"{scope_label}\n(pairs used: dark)")

    # ---- right: each animal's intact -> lesion parameter, paired t-test per group ----
    df_scope = df[df["fit_scope"] == scope]
    ax_left = fig.subplot(fig.gs[s, 1])
    ax_right = ax_left.twinx()

    for ax, params in ((ax_left, left_params), (ax_right, right_params)):
        stars = []
        for param in params:
            x = order.index(param)
            for g in groups:
                paired = (
                    df_scope[
                        (df_scope["param_names"] == param) & (df_scope["group"] == g)
                    ]
                    .pivot_table(index="name", columns="lesion", values="param_values")[
                        ["intact", lesion_tag]
                    ]
                    .dropna()
                )
                xs = [x + dodge[f"intact_{g}"], x + dodge[f"lesion_{g}"]]
                for _, row in paired.iterrows():
                    ax.plot(xs, row.values, color="0.6", lw=1.2, zorder=2)
                for xc, cond, tag in zip(
                    xs, ("intact", "lesion"), ("intact", lesion_tag)
                ):
                    color = palette[f"{cond}_{g}"]
                    ax.scatter(
                        np.full(len(paired), xc),
                        paired[tag],
                        s=55,
                        color=color if cond == "intact" else "white",
                        edgecolor=color,
                        linewidth=2,
                        zorder=3,
                        rasterized=True,  # vector scatter markers crash Affinity
                    )
                    ax.hlines(
                        paired[tag].mean(),
                        xc - 0.08,
                        xc + 0.08,
                        color=color,
                        lw=3,
                        zorder=4,
                    )
                if len(paired) >= 2:
                    p = ttest_rel(paired["intact"], paired[lesion_tag]).pvalue
                    stars.append((np.mean(xs), paired.to_numpy().max(), p_to_stars(p)))
                    print(f"{scope:9s} {param:8s} {g:8s} paired t-test p = {p:.3f}")
        # stars above each group's pair, after the data sets the y-range
        ymin, ymax = ax.get_ylim()
        pad = 0.05 * (ymax - ymin)
        for xc, top, label in stars:
            ax.text(xc, top + pad, label, ha="center", va="bottom", fontsize=12)
        ax.set_ylim(ymin, ymax + 3 * pad)

    ax_right.spines["right"].set_visible(True)
    ax_left.set_xlim(-0.6, len(order) - 0.4)
    ax_left.axvline(len(left_params) - 0.5, color="gray", ls=":", lw=1.5)
    ax_left.axhline(0, color="gray", lw=1, zorder=0)
    ax_left.set_xticks(range(len(order)), [param_labels[p] for p in order])
    ax_left.set_ylabel(f"{scope_label}\nLearning rate / bias")
    ax_right.set_ylabel("Persev. / beta")

# legend above the first parameter panel
n = df.groupby("group")["name"].nunique()
fig.fig.axes[1].legend(
    [
        Line2D(
            [],
            [],
            ls="",
            marker="o",
            ms=9,
            mfc="white" if c.startswith("lesion") else palette[c],
            mec=palette[c],
            mew=2,
        )
        for c in cohort_order
    ],
    [c.replace("_", " ") for c in cohort_order],
    ncol=4,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.02),
    title=f"Paired animals: unstruc n={n.get('unstruc', 0)}, struc n={n.get('struc', 0)}",
)

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "qlearn_sticky_params_8020_mPFC_lesion", format="png", dpi=300)
fig.savefig(figdir / "qlearn_sticky_params_8020_mPFC_lesion", format="svg")

### RNN-fit N48 session: Intact animals

In [ ]:
import mab_subjects
from mab_builders.build_all import build
from mab_builders.simulate import rnn_simulated_exps

# each animal's fitted N48 RNN simulated on its own block schedule (cached CSVs)
exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)
sim_exps = rnn_simulated_exps(exps, "rnn_fit_n48_session", n_jobs=6)
build(
    "sim_rnn_n48_perf_probability_matrix",
    "sim_rnn_n48_swp_probability_matrix",
    "sim_rnn_n48_swp_around_flip",
    exps=sim_exps,
    n_jobs=6,
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
kinds = {"animal": "Animals", "fit": "Fitted RNN"}


def animal_and_fit(animal_product, fit_product):
    """8020 intact animals and each animal's fitted N48 RNN (simulated on the
    animal's own block schedule), tagged by 'kind'."""
    return pd.concat(
        [
            select(
                getattr(gd, animal_product).latest, paradigm="8020", lesion="intact"
            ).assign(kind="animal"),
            select(
                getattr(gd, fit_product).latest, paradigm="8020", lesion="intact"
            ).assign(kind="fit"),
        ]
    )


def s_minus_u_grids(df, mat_col):
    """struc - unstruc grid per kind, on probability axes shared by both kinds."""
    df = df.assign(group=df["group"] + "_" + df["kind"])
    probs, means, counts = prob_grid_group_means(
        df, mat_col, groups=[f"{g}_{k}" for k in kinds for g in hue_order]
    )
    diffs = {k: means[f"struc_{k}"] - means[f"unstruc_{k}"] for k in kinds}
    seen = np.zeros(len(probs), dtype=bool)
    for d in diffs.values():
        seen |= np.isfinite(d).any(axis=0) | np.isfinite(d).any(axis=1)
    diffs = {k: d[np.ix_(seen, seen)] for k, d in diffs.items()}
    return probs[seen], diffs, counts


# prob grids: P(High) over the last 90 trials, P(Switch) over trials 2-100
perf_grids = s_minus_u_grids(
    animal_and_fit(
        "abstract_perf_probability_matrix", "sim_rnn_n48_perf_probability_matrix"
    ),
    "perf_mat",
)
swp_grids = s_minus_u_grids(
    animal_and_fit(
        "abstract_swp_probability_matrix", "sim_rnn_n48_swp_probability_matrix"
    ),
    "swp_mat",
)

# flip response: P(Switch | flipped) - P(Switch | same), from the previous block's
# last 10 trials into the new block's first 50
flip_df = animal_and_fit("abstract_swp_around_flip", "sim_rnn_n48_swp_around_flip")
wide = flip_df.pivot_table(
    index=["kind", "group", "name", "trial_id"],
    columns="prev_best_arm",
    values="switch_prob",
)
resp_df = (wide["flipped"] - wide["same"]).rename("flip_resp").reset_index()

# each subject's peak flip response after the transition (gaussian-smoothed), as in
# the Cosyne abstract: its trial and height
sigma = 2  # smoothing (trials)
rows = []
for (kind, group, name), d in resp_df[resp_df["trial_id"] > 0].groupby(
    ["kind", "group", "name"]
):
    smooth = gaussian_filter1d(np.nan_to_num(d["flip_resp"].to_numpy()), sigma)
    rows.append(
        dict(
            kind=kinds[kind],
            group=group,
            name=name,
            peak_trial=d["trial_id"].to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 4, size=(17, 9), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- top row: struc - unstruc prob grids, animals vs RNNs ----
for col, (name, (probs, diffs, counts), mat_col, label) in enumerate(
    [
        ("P(High)", perf_grids, "perf_mat", "Diff. P(High)"),
        ("P(Switch)", swp_grids, "swp_mat", "Diff. P(Switch)"),
    ]
):
    # shared color scale so the animal and fitted-RNN struc advantage are comparable
    colors = diverging_bins(
        np.nanmax([np.nanmax(np.abs(d)) for d in diffs.values()]),
        step=BIN_STEPS[mat_col],
    )
    axes = []
    for i, (k, kind_label) in enumerate(kinds.items()):
        ax = fig.subplot(fig.gs[0, 2 * col + i])
        im = draw_prob_grid(
            ax,
            diffs[k],
            probs,
            has_diag=np.isfinite(np.diag(diffs[k])).any(),  # P(High) has none
            **colors,
        )
        ax.set_title(
            f"{name}: {kind_label}\nstruc − unstruc "
            f"(n={counts[f'struc_{k}']} vs {counts[f'unstruc_{k}']})"
        )
        if i > 0:
            ax.set_ylabel("")
        axes.append(ax)
    fig.fig.colorbar(im, ax=axes, label=label, shrink=0.8)

# ---- bottom row: P(Switch) around the transition (rows: animals, RNNs; columns:
# raw flipped, raw same, flipped - same), then peak height and peak trial ----
curve_gs = fig.gs[1, 0:2].subgridspec(2, 3)
curve_cols = [
    ("flipped", flip_df[flip_df["prev_best_arm"] == "flipped"], "switch_prob"),
    ("same", flip_df[flip_df["prev_best_arm"] == "same"], "switch_prob"),
    ("flipped − same", resp_df, "flip_resp"),
]
for r, (k, kind_label) in enumerate(kinds.items()):
    row_axes = []
    for j, (title, data, y) in enumerate(curve_cols):
        ax = fig.subplot(curve_gs[r, j])
        sns.lineplot(
            data=data[data["kind"] == k],
            x="trial_id",
            y=y,
            hue="group",
            hue_order=hue_order,
            palette=palette,
            lw=2,
            err_kws=dict(edgecolor="none", alpha=0.3),
            errorbar="se",
            legend=False,
            ax=ax,
        )
        ax.axvline(0, color="gray", ls=":", lw=1.2)  # block transition
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        ax.set_xlim(-10, 50)
        ax.set_xticks([-10, 20, 50])  # transition marked by the dotted line
        ax.set_xlabel("Trials from transition" if r == 1 else "")
        ax.set_ylabel(f"{kind_label}\nP(Switch)" if j == 0 else "")
        if r == 0:
            ax.set_title(title)
        row_axes.append(ax)

    # raw flipped and same share a y-scale; ticks at 0, middle, top
    hi = max(ax.dataLim.y1 for ax in row_axes[:2])
    for ax in row_axes[:2]:
        three_yticks(ax, lo=0.0, hi=hi)
    ax = row_axes[2]
    ax.axhline(0, color="gray", lw=1)
    top = three_yticks(ax, lo=0.0)[-1]
    ax.set_ylim(-0.1 * top, top)  # room for small dips below 0


def plot_peak_box(ax, data, y, ylabel, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=data,
            x="kind",
            y=y,
            hue="group",
            hue_order=hue_order,
            ax=ax,
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=5,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)


# peak height: each kind gets its own y-axis (fitted RNNs switch more than animals)
height_gs = fig.gs[1, 2].subgridspec(1, 2)
for i, kind_label in enumerate(kinds.values()):
    plot_peak_box(
        fig.subplot(height_gs[i]),
        peaks[peaks["kind"] == kind_label],
        "peak_swp",
        "Flip response\npeak Diff. P(Switch)" if i == 0 else "",
        lambda ax: three_yticks(ax, lo=0.0),
    )

# peak trial: same trial scale for animals and fitted RNNs
plot_peak_box(
    fig.subplot(fig.gs[1, 3]),
    peaks,
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)

print(peaks.groupby(["kind", "group"])[["peak_trial", "peak_swp"]].median())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "animal_fitrnn_n48_8020", format="png", dpi=300)
fig.savefig(figdir / "animal_fitrnn_n48_8020", format="svg")

### RNN-fit N48 window: Intact animals

In [ ]:
import mab_subjects
from mab_builders.build_all import build
from mab_builders.simulate import rnn_simulated_exps

# each animal's fitted N48 RNN, hidden state reset at window starts (1000 epochs),
# simulated on its own block schedule (cached CSVs)
exps = (
    mab_subjects.unstruc.p8020_good_intact_sess
    + mab_subjects.struc.p8020_good_intact_sess
)
sim_exps = rnn_simulated_exps(exps, "rnn_fit_n48_window", n_jobs=6)
build(
    "sim_rnn_n48win_perf_probability_matrix",
    "sim_rnn_n48win_swp_probability_matrix",
    "sim_rnn_n48win_swp_around_flip",
    exps=sim_exps,
    n_jobs=6,
)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]
kinds = {"animal": "Animals", "fit": "Fitted RNN (window)"}


def animal_and_fit(animal_product, fit_product):
    """8020 intact animals and each animal's fitted N48 window-reset RNN
    (simulated on the animal's own block schedule), tagged by 'kind'."""
    return pd.concat(
        [
            select(
                getattr(gd, animal_product).latest, paradigm="8020", lesion="intact"
            ).assign(kind="animal"),
            select(
                getattr(gd, fit_product).latest, paradigm="8020", lesion="intact"
            ).assign(kind="fit"),
        ]
    )


def s_minus_u_grids(df, mat_col):
    """struc - unstruc grid per kind, on probability axes shared by both kinds."""
    df = df.assign(group=df["group"] + "_" + df["kind"])
    probs, means, counts = prob_grid_group_means(
        df, mat_col, groups=[f"{g}_{k}" for k in kinds for g in hue_order]
    )
    diffs = {k: means[f"struc_{k}"] - means[f"unstruc_{k}"] for k in kinds}
    seen = np.zeros(len(probs), dtype=bool)
    for d in diffs.values():
        seen |= np.isfinite(d).any(axis=0) | np.isfinite(d).any(axis=1)
    diffs = {k: d[np.ix_(seen, seen)] for k, d in diffs.items()}
    return probs[seen], diffs, counts


# prob grids: P(High) over the last 90 trials, P(Switch) over trials 2-100
perf_grids = s_minus_u_grids(
    animal_and_fit(
        "abstract_perf_probability_matrix", "sim_rnn_n48win_perf_probability_matrix"
    ),
    "perf_mat",
)
swp_grids = s_minus_u_grids(
    animal_and_fit(
        "abstract_swp_probability_matrix", "sim_rnn_n48win_swp_probability_matrix"
    ),
    "swp_mat",
)

# flip response: P(Switch | flipped) - P(Switch | same), from the previous block's
# last 10 trials into the new block's first 50
flip_df = animal_and_fit("abstract_swp_around_flip", "sim_rnn_n48win_swp_around_flip")
wide = flip_df.pivot_table(
    index=["kind", "group", "name", "trial_id"],
    columns="prev_best_arm",
    values="switch_prob",
)
resp_df = (wide["flipped"] - wide["same"]).rename("flip_resp").reset_index()

# each subject's peak flip response after the transition (gaussian-smoothed), as in
# the Cosyne abstract: its trial and height
sigma = 2  # smoothing (trials)
rows = []
for (kind, group, name), d in resp_df[resp_df["trial_id"] > 0].groupby(
    ["kind", "group", "name"]
):
    smooth = gaussian_filter1d(np.nan_to_num(d["flip_resp"].to_numpy()), sigma)
    rows.append(
        dict(
            kind=kinds[kind],
            group=group,
            name=name,
            peak_trial=d["trial_id"].to_numpy()[np.argmax(smooth)],
            peak_swp=smooth.max(),
        )
    )
peaks = pd.DataFrame(rows)

fig = plotting.Fig(2, 4, size=(17, 9), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- top row: struc - unstruc prob grids, animals vs RNNs ----
for col, (name, (probs, diffs, counts), mat_col, label) in enumerate(
    [
        ("P(High)", perf_grids, "perf_mat", "Diff. P(High)"),
        ("P(Switch)", swp_grids, "swp_mat", "Diff. P(Switch)"),
    ]
):
    # shared color scale so the animal and fitted-RNN struc advantage are comparable
    colors = diverging_bins(
        np.nanmax([np.nanmax(np.abs(d)) for d in diffs.values()]),
        step=BIN_STEPS[mat_col],
    )
    axes = []
    for i, (k, kind_label) in enumerate(kinds.items()):
        ax = fig.subplot(fig.gs[0, 2 * col + i])
        im = draw_prob_grid(
            ax,
            diffs[k],
            probs,
            has_diag=np.isfinite(np.diag(diffs[k])).any(),  # P(High) has none
            **colors,
        )
        ax.set_title(
            f"{name}: {kind_label}\nstruc − unstruc "
            f"(n={counts[f'struc_{k}']} vs {counts[f'unstruc_{k}']})"
        )
        if i > 0:
            ax.set_ylabel("")
        axes.append(ax)
    fig.fig.colorbar(im, ax=axes, label=label, shrink=0.8)

# ---- bottom row: P(Switch) around the transition (rows: animals, RNNs; columns:
# raw flipped, raw same, flipped - same), then peak height and peak trial ----
curve_gs = fig.gs[1, 0:2].subgridspec(2, 3)
curve_cols = [
    ("flipped", flip_df[flip_df["prev_best_arm"] == "flipped"], "switch_prob"),
    ("same", flip_df[flip_df["prev_best_arm"] == "same"], "switch_prob"),
    ("flipped − same", resp_df, "flip_resp"),
]
for r, (k, kind_label) in enumerate(kinds.items()):
    row_axes = []
    for j, (title, data, y) in enumerate(curve_cols):
        ax = fig.subplot(curve_gs[r, j])
        sns.lineplot(
            data=data[data["kind"] == k],
            x="trial_id",
            y=y,
            hue="group",
            hue_order=hue_order,
            palette=palette,
            lw=2,
            err_kws=dict(edgecolor="none", alpha=0.3),
            errorbar="se",
            legend=False,
            ax=ax,
        )
        ax.axvline(0, color="gray", ls=":", lw=1.2)  # block transition
        ax.grid(axis="y", zorder=-1, alpha=0.5)
        ax.set_xlim(-10, 50)
        ax.set_xticks([-10, 20, 50])  # transition marked by the dotted line
        ax.set_xlabel("Trials from transition" if r == 1 else "")
        ax.set_ylabel(f"{kind_label}\nP(Switch)" if j == 0 else "")
        if r == 0:
            ax.set_title(title)
        row_axes.append(ax)

    # raw flipped and same share a y-scale; ticks at 0, middle, top
    hi = max(ax.dataLim.y1 for ax in row_axes[:2])
    for ax in row_axes[:2]:
        three_yticks(ax, lo=0.0, hi=hi)
    ax = row_axes[2]
    ax.axhline(0, color="gray", lw=1)
    top = three_yticks(ax, lo=0.0)[-1]
    ax.set_ylim(-0.1 * top, top)  # room for small dips below 0


def plot_peak_box(ax, data, y, ylabel, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=data,
            x="kind",
            y=y,
            hue="group",
            hue_order=hue_order,
            ax=ax,
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=5,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)


# peak height: each kind gets its own y-axis (fitted RNNs switch more than animals)
height_gs = fig.gs[1, 2].subgridspec(1, 2)
for i, kind_label in enumerate(kinds.values()):
    plot_peak_box(
        fig.subplot(height_gs[i]),
        peaks[peaks["kind"] == kind_label],
        "peak_swp",
        "Flip response\npeak Diff. P(Switch)" if i == 0 else "",
        lambda ax: three_yticks(ax, lo=0.0),
    )

# peak trial: same trial scale for animals and fitted RNNs
plot_peak_box(
    fig.subplot(fig.gs[1, 3]),
    peaks,
    "peak_trial",
    "Flip response\npeak (trial)",
    lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
)

print(peaks.groupby(["kind", "group"])[["peak_trial", "peak_swp"]].median())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "animal_fitrnn_n48win_8020", format="png", dpi=300)
fig.savefig(figdir / "animal_fitrnn_n48win_8020", format="svg")

### S - U grids and flip response: 8020-intact animals vs fitted RNN N48 (session and window resets)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from neuropy import plotting
import mab_subjects
from mab_builders.select import select
from mab_colors import Palette2Arm
from mab_abstract.mab_abstract_plotters import (
    three_yticks,
    BIN_STEPS,
    prob_grid_group_means,
    draw_prob_grid,
    diverging_bins,
)
from statplotannot.plots import SeabornPlotter

gd = mab_subjects.GroupData()
palette = Palette2Arm().as_dict()
palette_dots = Palette2Arm(lightness_scale=0.7).as_dict()  # darker shade of each box
hue_order = ["unstruc", "struc"]

# rows: animals and each animal's fitted N48 RNN (hidden state reset per session or
# per window) simulated on its own block schedule; product name prefixes per source
sources = {
    "Animals": dict(
        perf="abstract_perf_probability_matrix",
        swp="abstract_swp_probability_matrix",
        flip="abstract_swp_around_flip",
    ),
    "RNN N48\nsession": dict(
        perf="sim_rnn_n48_perf_probability_matrix",
        swp="sim_rnn_n48_swp_probability_matrix",
        flip="sim_rnn_n48_swp_around_flip",
    ),
    "RNN N48\nwindow": dict(
        perf="sim_rnn_n48win_perf_probability_matrix",
        swp="sim_rnn_n48win_swp_probability_matrix",
        flip="sim_rnn_n48win_swp_around_flip",
    ),
}


def load(product):
    """8020 intact rows of a product (fitted-RNN rows carry the animal's tags)."""
    return select(getattr(gd, product).latest, paradigm="8020", lesion="intact")


def s_minus_u(product, mat_col):
    """struc - unstruc mean grid, with its probabilities and group counts."""
    probs, means, counts = prob_grid_group_means(load(product), mat_col)
    return probs, means["struc"] - means["unstruc"], counts


# ---- data: S - U grids, flip response curves and peaks for every source ----
grids = {m: {} for m in ("perf", "swp")}
resp, peaks = [], []
sigma = (
    2  # gaussian smoothing (trials) of each flip response, as in the Cosyne abstract
)
for src, products in sources.items():
    grids["perf"][src] = s_minus_u(products["perf"], "perf_mat")
    grids["swp"][src] = s_minus_u(products["swp"], "swp_mat")

    # flip response: P(Switch | flipped) - P(Switch | same), previous block's last
    # 10 trials into the new block's first 50
    flip = load(products["flip"])
    wide = flip.pivot_table(
        index=["group", "name", "trial_id"],
        columns="prev_best_arm",
        values="switch_prob",
    )
    r = (wide["flipped"] - wide["same"]).rename("flip_resp").reset_index()
    resp.append(r.assign(source=src))
    for (group, name), d in r[r["trial_id"] > 0].groupby(["group", "name"]):
        smooth = gaussian_filter1d(np.nan_to_num(d["flip_resp"].to_numpy()), sigma)
        peaks.append(
            dict(
                source=src,
                kind="",  # single x category per panel
                group=group,
                name=name,
                peak_trial=d["trial_id"].to_numpy()[np.argmax(smooth)],
                peak_swp=smooth.max(),
            )
        )
resp = pd.concat(resp, ignore_index=True)
peaks = pd.DataFrame(peaks)

fig = plotting.Fig(3, 5, size=(21, 13), fontsize=14, axis_lw=1.5, tick_size=5)

# ---- columns 1-2: struc - unstruc prob grids, one colour scale per measure ----
for col, (m, mat_col, title, label) in enumerate(
    [
        ("perf", "perf_mat", "P(High)", "Diff. P(High)"),
        ("swp", "swp_mat", "P(Switch)", "Diff. P(Switch)"),
    ]
):
    colors = diverging_bins(
        np.nanmax([np.nanmax(np.abs(d)) for _, d, _ in grids[m].values()]),
        step=BIN_STEPS[mat_col],
    )
    axes = []
    for r, (src, (probs, diff, counts)) in enumerate(grids[m].items()):
        ax = fig.subplot(fig.gs[r, col])
        im = draw_prob_grid(
            ax, diff, probs, has_diag=np.isfinite(np.diag(diff)).any(), **colors
        )
        ax.set_title(
            f"{title}: struc − unstruc\n(n={counts['struc']} vs {counts['unstruc']})"
            if r == 0
            else f"(n={counts['struc']} vs {counts['unstruc']})"
        )
        if col == 0:
            ax.set_ylabel(f"{src}\n\nLower p")
        else:
            ax.set_ylabel("")
        if r < 2:
            ax.set_xlabel("")
        axes.append(ax)
    # horizontal, under the column, so it doesn't crowd the next column
    cb = fig.fig.colorbar(
        im, ax=axes, label=label, location="bottom", shrink=0.8, aspect=30
    )
    edges = im.norm.boundaries  # binned colours: tick the end blocks' centres and 0
    end = (edges[-2] + edges[-1]) / 2
    cb.set_ticks([-end, 0, end])
    cb.ax.minorticks_off()

# ---- column 3: flip response curves, shared y-scale ----
curve_axes = []
for r, src in enumerate(sources):
    ax = fig.subplot(fig.gs[r, 2])
    sns.lineplot(
        data=resp[resp["source"] == src],
        x="trial_id",
        y="flip_resp",
        hue="group",
        hue_order=hue_order,
        palette=palette,
        lw=2.5,
        err_kws=dict(edgecolor="none", alpha=0.3),
        errorbar="se",
        legend=False,
        ax=ax,
    )
    ax.axvline(0, color="gray", ls=":", lw=1.5)  # block transition
    ax.axhline(0, color="gray", lw=1)
    ax.grid(axis="y", zorder=-1, alpha=0.5)
    ax.set_xlim(-10, 50)
    ax.set_xticks([-10, 0, 25, 50])
    ax.set_xlabel("Trials from transition" if r == 2 else "")
    ax.set_ylabel("Flip response\nDiff. P(Switch)")
    if r == 0:
        ax.set_title("Flip response\n(flipped − same)")
    curve_axes.append(ax)
hi = max(ax.dataLim.y1 for ax in curve_axes)
for ax in curve_axes:  # labelled ticks at 0, middle, top; room for small dips below 0
    top = three_yticks(ax, lo=0.0, hi=hi)[-1]
    ax.set_ylim(-0.1 * top, top)


# ---- columns 4-5: flip response peak height and trial, boxplot + Mann-Whitney ----
def plot_peak_box(ax, data, y, set_yaxis):
    plotter = (
        SeabornPlotter(
            data=data, x="kind", y=y, hue="group", hue_order=hue_order, ax=ax
        )
        .boxplot_filled(palette=palette, sep=0.4)
        .stripplot(
            palette=palette_dots,
            alpha=0.8,
            size=6,
            jitter=0.03,
            offset=0.1,  # dots beside, not on, their box
            edgecolor="w",
            rasterized=True,  # vector scatter markers crash Affinity
        )
    )
    set_yaxis(ax)  # before the brackets (data units)
    plotter.stat_classical(
        "Mann-Whitney",
        pvalue_thresholds=[[0.001, "***"], [0.01, "**"], [0.05, "*"], [1, "ns"]],
        loc="outside",
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    ax.set_xlabel("")
    ax.set_xticks([])


height_hi = peaks["peak_swp"].max()
for r, src in enumerate(sources):
    data = peaks[peaks["source"] == src]
    ax = fig.subplot(fig.gs[r, 3])
    plot_peak_box(
        ax, data, "peak_swp", lambda ax: three_yticks(ax, lo=0.0, hi=height_hi)
    )
    ax.set_ylabel("Peak height\nDiff. P(Switch)")
    if r == 0:
        ax.set_title("Flip response\npeak height", pad=30)  # above the bracket
    ax = fig.subplot(fig.gs[r, 4])
    plot_peak_box(
        ax,
        data,
        "peak_trial",
        lambda ax: (ax.set_ylim(0, 50), ax.set_yticks([0, 25, 50])),
    )
    ax.set_ylabel("Peak trial")
    if r == 0:
        ax.set_title("Flip response\npeak trial", pad=30)

print(peaks.groupby(["source", "group"])[["peak_trial", "peak_swp"]].median())

# PNG for pasting into PowerPoint, SVG if it needs editing
figdir = mab_subjects.FigPath.presentations
fig.savefig(figdir / "s_minus_u_flip_animal_vs_rnnfit_8020", format="png", dpi=300)
fig.savefig(figdir / "s_minus_u_flip_animal_vs_rnnfit_8020", format="svg")